In [1]:
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
import pandas as pd

import os
import warnings 
warnings.filterwarnings('ignore')

C:\Users\Hp\Documents\Projects\apartments_for_rent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_parquet('../data/cleaned_data.parquet', engine='fastparquet')

In [ ]:
df.columns

In [ ]:
# ==========================================
# STEP 1: Initialize Persistent ChromaDB
# ==========================================
chroma_client = chromadb.PersistentClient(path="./housing_vector_db")


embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name="property_recommendations", 
    embedding_function=embedding_func
)

In [6]:
# ==========================================
# STEP 2: Prepare & Clean Metadata for Chroma
# ==========================================
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median()).astype(int)
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce').fillna(df['bathrooms'].median()).astype(int)
df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(df['price'].median()).astype(int)

metadatas = df[['category', 'cityname', 'price', 'bedrooms', 'bathrooms']].to_dict(orient='records')
documents = df['body'].astype(str).tolist()
ids = df['id'].astype(str).tolist() 

In [8]:
# ==========================================
# STEP 3: Add Data to Vector Database (Safe Chunking)
# ==========================================
print("Adding data to ChromaDB in safe batches...")

# Define a safe batch size well below the 5,461 limit
BATCH_SIZE = 2000  
total_rows = len(ids)

for i in range(0, total_rows, BATCH_SIZE):
    batch_ids = ids[i:i + BATCH_SIZE]
    batch_documents = documents[i:i + BATCH_SIZE]
    batch_metadatas = metadatas[i:i + BATCH_SIZE]
    
    print(f"-> Uploading batch {i // BATCH_SIZE + 1} (Rows {i} to {min(i + BATCH_SIZE, total_rows)})...")
    
    # Add this chunk to the collection
    collection.add(
        ids=batch_ids,
        documents=batch_documents,
        metadatas=batch_metadatas
    )

print("Database populated successfully!")

Adding data to ChromaDB in safe batches...
-> Uploading batch 1 (Rows 0 to 2000)...
-> Uploading batch 2 (Rows 2000 to 4000)...
-> Uploading batch 3 (Rows 4000 to 6000)...
-> Uploading batch 4 (Rows 6000 to 8000)...
-> Uploading batch 5 (Rows 8000 to 10000)...
Database populated successfully!


In [9]:
def chromadb_recommend(query_text, max_price=3000, min_beds=2, top_n=5):
    """
    Performs true deep semantic search combined with structural database constraints.
    """
    print(f"🔍 Searching for: '{query_text}' with Price <= ${max_price} and Beds >= {min_beds}...\n")
    
    # Run the query directly against the vector database
    results = collection.query(
        query_texts=[query_text],
        n_results=top_n,
        where={
            "$and": [
                {"price": {"$lte": max_price}},
                {"bedrooms": {"$gte": min_beds}}
            ]
        }
    )
    
    # Re-structure the output back into a pretty Pandas DataFrame for display
    recommendations = []
    for i in range(len(results['ids'][0])):
        rec = {
            "id": results['ids'][0][i],
            "body_snippet": results['documents'][0][i][:100] + "...",
            "distance_score": results['distances'][0][i], # Lower means closer match
            **results['metadatas'][0][i]
        }
        recommendations.append(rec)
        
    return pd.DataFrame(recommendations)

In [10]:
# Look for a modern vibe, but strictly enforce a budget under $2,500 and at least 2 bedrooms
chromadb_recommend(
    query_text="modern apartment with a pool and balcony", 
    max_price=2500, 
    min_beds=2, 
    top_n=5
)

🔍 Searching for: 'modern apartment with a pool and balcony' with Price <= $2500 and Beds >= 2...



,id,body_snippet,distance_score,bathrooms,cityname,price,bedrooms,category
0,5508942181,"Square footage: 1200 square ft, unit number: 4...",0.385701,2,Windermere,1540,2,housing/rent/apartment
1,5508859939,Silver. Shared spaces powered by 100% renewabl...,0.394892,1,Dallas,1207,2,housing/rent/apartment
2,5509038769,"A/c Window, Balcony, Ceiling Fan, Dishwasher, ...",0.398998,1,Colorado Springs,1155,2,housing/rent/apartment
3,5508786668,"Our spacious one, 2 and 3 beds apartment homes...",0.409076,1,Capitol Heights,1480,3,housing/rent/apartment
4,5508867739,"Square footage: 1036 square feet, unit number:...",0.424869,2,Clermont,1320,2,housing/rent/apartment
